## بوابة التدريب الإلزامية

قبل أي تنزيل أو تدريب، أضف في Colab Secrets القيم التالية: `DRPLANT_TRAINING_PROFILE`, `DRPLANT_TRAINING_PROFILE_SIGNATURE`, `DRPLANT_TRAINING_TRUSTED_PUBLIC_KEY`. يجب أن تشير إلى profile وتوقيعه والمفتاح العام الموثوق. سيوقف الدفتر التنفيذ إذا غابت أي قيمة أو فشل فحص البيانات. لا تتجاوز هذه الخلية.


In [ ]:
import os
from google.colab import userdata
_REQUIRED_DRPLANT_GATE_SECRETS = (
    "DRPLANT_TRAINING_PROFILE",
    "DRPLANT_TRAINING_PROFILE_SIGNATURE",
    "DRPLANT_TRAINING_TRUSTED_PUBLIC_KEY",
)
_missing = []
for _name in _REQUIRED_DRPLANT_GATE_SECRETS:
    _value = userdata.get(_name)
    if not _value:
        _missing.append(_name)
    else:
        os.environ[_name] = _value
if _missing:
    raise RuntimeError("NO-GO: configure these Colab Secrets before running any training cell: " + ", ".join(_missing))
print("Dr.Plant training gate credentials loaded; the training script will verify the signed profile and data before training.")


# Dr Plant — Full Multi-crop Training v2

**الإصدار الثاني يستخدم كامل مجموعة PlantVillage: 54,304 صورة و38 فئة ضمن 14 محصولاً.**

لا يستخدم عينة محدودة. يحافظ على اختبار المصدر كما هو، ويقتطع مجموعة تحقق من التدريب باستخدام `StratifiedGroupKFold` مع عزل `leaf_id`. يحفظ checkpoints ونتائج ONNX. لا يستبدل نموذج الليمون الحالي.

> هذه بيانات صور مجمعة وليست تحققاً حقلياً محلياً. يلزم تدقيق التسميات وترخيص البيانات قبل الاعتماد التجاري أو التشخيصي.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, csv, shutil, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/DrPlant/multicrop-plantvillage-v2')
    print('Persistent Google Drive output enabled:', ROOT)
except Exception as e:
    ROOT = Path('/content/DrPlant/multicrop-plantvillage-v2')
    print('Drive unavailable; using temporary Colab disk:', repr(e))
ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
!pip -q install "datasets>=3.0.0" scikit-learn pandas pillow onnx onnxruntime
if not Path('/content/dr-plant').exists():
    !git clone --branch research/morphology-v2 https://github.com/jonbander/dr-plant.git /content/dr-plant
%cd /content/dr-plant
!git pull


## 1. تحميل وفحص كامل البيانات
نتوقع 54,304 صورة و38 فئة. يوقف الدفتر إذا اختلف العدد أو نقصت الفئات.

In [ ]:
from datasets import load_dataset
from collections import Counter, defaultdict
D = load_dataset('fatemehENG/plantvillage-full', split='train')
classes = sorted(set(D['class_label']))
print('Total images:', len(D), '| classes:', len(classes))
print('Source splits:', Counter(D['split']))
assert len(D) == 54304 and len(classes) == 38, 'Dataset version/count changed; inspect before training.'


## 2. تصدير الصور كاملة وتقسيم مجموعات الأوراق
لا يوجد سقف لعدد الصور لكل فئة. يبقى اختبار المصدر كاملاً، ونقسم التدريب إلى تدريب/تحقق بطريقة مجموعات الأوراق.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from PIL import Image
from collections import Counter
IMAGE_ROOT = ROOT / 'images'; IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
source_train=[i for i,s in enumerate(D['split']) if s=='train']; source_test=[i for i,s in enumerate(D['split']) if s=='test']
y=np.array([D[i]['class_label'] for i in source_train]); g=np.array([D[i]['leaf_id'] for i in source_train])
sg=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=2026)
tr_rel,val_rel=next(sg.split(np.zeros(len(y)),y,g))
split_indices={'train':{source_train[i] for i in tr_rel},'validation':{source_train[i] for i in val_rel},'test':set(source_test)}
groups={s:{D[i]['leaf_id'] for i in ix} for s,ix in split_indices.items()}
for a,b in [('train','validation'),('train','test'),('validation','test')]: assert not (groups[a]&groups[b]),f'Group leakage {a}/{b}'
for s,ix in split_indices.items(): assert {D[i]['class_label'] for i in ix}==set(classes),f'Missing class in {s}'
manifest=ROOT/'manifest.csv'; counts=Counter()
with manifest.open('w',newline='',encoding='utf-8') as f:
 fields=['relative_path','class_label','split','group_id_for_split','provenance_verified','expert_label_review','near_duplicate_review']; w=csv.DictWriter(f,fieldnames=fields); w.writeheader()
 for split,ix in split_indices.items():
  for i in sorted(ix):
   r=D[i]; label=r['class_label']; folder=IMAGE_ROOT/label; folder.mkdir(parents=True,exist_ok=True); path=folder/f'{split}_{i:06d}.jpg'
   if not path.exists(): r['image'].convert('RGB').save(path,'JPEG',quality=90,optimize=True)
   w.writerow({'relative_path':str(path.relative_to(IMAGE_ROOT)),'class_label':label,'split':split,'group_id_for_split':r['leaf_id'],'provenance_verified':'yes','expert_label_review':'no','near_duplicate_review':'no'}); counts[split]+=1
print('Exported complete dataset:',dict(counts),'total:',sum(counts.values()))


## 3. تدريب كامل البيانات
على GPU في Colab يمكن رفع epochs إلى 10–20. يحفظ السكربت checkpoint بعد كل epoch ويمكن استئناف التدريب بـ `--resume`.

In [ ]:
OUT=ROOT/'model-output'; OUT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'scripts/train_multicrop_full_v2.py',str(IMAGE_ROOT),str(manifest),'--out',str(OUT),'--epochs','12','--batch-size','32','--learning-rate','0.0003','--pretrained','--image-size','224','--workers','2','--exploratory-non-independent','--unfreeze-last-blocks','2','--training-profile',os.environ['DRPLANT_TRAINING_PROFILE'],'--training-profile-signature',os.environ['DRPLANT_TRAINING_PROFILE_SIGNATURE'],'--training-data-root',str(IMAGE_ROOT),'--training-trusted-public-key',os.environ['DRPLANT_TRAINING_TRUSTED_PUBLIC_KEY']]
print('Running:', ' '.join(cmd)); subprocess.run(cmd,check=True)


## 4. تصدير ONNX والتحقق العددي

In [ ]:
import torch, onnx, onnxruntime as ort, numpy as np
from torchvision import models
from torch import nn
metrics=json.loads((OUT/'metrics.json').read_text()); labels=metrics['classes']
ck=torch.load(OUT/'best-model.pt',map_location='cpu',weights_only=False)
model=models.mobilenet_v3_small(weights=None); model.classifier[3]=nn.Linear(model.classifier[3].in_features,len(labels)); model.load_state_dict(ck['state_dict']); model.eval()
onnx_path=OUT/'plantvillage-full-multicrop-mobilenetv3-small.onnx'; dummy=torch.randn(1,3,224,224)
torch.onnx.export(model,dummy,str(onnx_path),input_names=['input'],output_names=['logits'],dynamic_axes={'input':{0:'batch'},'logits':{0:'batch'}},opset_version=17,dynamo=False)
onnx.checker.check_model(onnx.load(str(onnx_path))); sess=ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider'])
with torch.no_grad(): pt=model(dummy).numpy()
ox=sess.run(['logits'],{'input':dummy.numpy()})[0]; max_diff=float(np.max(np.abs(pt-ox))); assert max_diff<1e-3
(OUT/'labels.json').write_text(json.dumps(labels,ensure_ascii=False,indent=2)+'\n')
summary={'dataset':'fatemehENG/plantvillage-full','total_images':54304,'classes':len(labels),'images_per_split':metrics['images_per_split'],'best_validation_accuracy':metrics['best_validation_accuracy'],'test_accuracy':metrics['test_accuracy'],'test_macro_f1':metrics['test_macro_f1'],'onnx_max_abs_difference':max_diff,'group_isolation_verified':True,'expert_label_review':'not completed','field_validation':'not completed'}
(OUT/'run-summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n'); print(json.dumps(summary,ensure_ascii=False,indent=2))


## 5. حفظ النتائج
إذا كان Drive متصلاً، تكون النتائج في `MyDrive/DrPlant/multicrop-plantvillage-v2`. احتفظ بـ `training-checkpoint.pt` للاستئناف.